# Week 05 — BBO capstone driver

Round 5. Four observations per function — enough to fit a GP, though with 4 points in up to 8 dimensions the posterior is still almost entirely prior.

**This round moves every function into fresh mid-cube territory.** The diagonal hypothesis died in W4 and the W2/W3 regions are either exhausted or poor, so the priority is coverage rather than refinement.

The GP is fitted and inspected below, but the proposals are chosen by hand. With this much data that is the honest thing to do — and the trust check in section 2 shows why.

In [ ]:
%matplotlib inline
import os, sys, warnings
warnings.filterwarnings("ignore")
# Walk up until bbo.py is found, so the notebook runs from anywhere in the repo.
_root = os.getcwd()
while not os.path.exists(os.path.join(_root, "bbo.py")) and os.path.dirname(_root) != _root:
    _root = os.path.dirname(_root)
os.chdir(_root); sys.path.insert(0, _root)
import numpy as np
import pandas as pd
import bbo

WEEK = 5
PRIOR = WEEK - 1          # data state this round was proposed from
SEED = 5
OUTDIR = f"outputs/week{WEEK:02d}"; os.makedirs(OUTDIR, exist_ok=True)

# What each function is getting this round, and why.
PLAN = {
    1: 'relocate to fresh mid-cube region',
    2: 'relocate to fresh mid-cube region',
    3: 'relocate to fresh mid-cube region',
    4: 'relocate to fresh mid-cube region',
    5: 'relocate to fresh mid-cube region',
    6: 'relocate to fresh mid-cube region',
    7: 'relocate to fresh mid-cube region',
    8: 'relocate to fresh mid-cube region',
}
pd.DataFrame([dict(func=f"F{f}", d=bbo.DIMS[f], move=PLAN[f]) for f in bbo.FUNC_IDS])


## 1. Data — the state this round was proposed from

Best point on record per function, truncated to rounds ≤ 4. Nothing below this cell may look at later rounds.


In [ ]:
# The ledger comes FIRST every round: the best point on record, not the latest one.
led = bbo.ledger(up_to=PRIOR)
led["best"] = led["best"].map(lambda v: f"{v:.6g}")
led["x"] = led["x"].map(bbo.submission)
led


## 2. Proposals — fresh regions, GP fitted but not trusted

The EI proposal is computed alongside for comparison. Where LOO R² is negative the two should be expected to disagree, and the GP's should be discarded.

In [ ]:
proposals = {
    1: np.array([0.048421, 0.137247]),
    2: np.array([0.048699, 0.137537]),
    3: np.array([0.489658, 0.159666, 0.895963]),
    4: np.array([0.569121, 0.44372, 0.402671, 0.283008]),
    5: np.array([0.275418, 0.703962, 0.058314, 0.62974]),
    6: np.array([0.448901, 0.916285, 0.331657, 0.205493, 0.612847]),
    7: np.array([0.659037, 0.842715, 0.27649, 0.501628, 0.787126, 0.394208]),
    8: np.array([0.015899, 0.258569, 0.158963, 0.325259, 0.785858, 0.212159, 0.958585, 0.071899]),
}

# What would EI have proposed, given the same data?
rows = []
for fid in bbo.FUNC_IDS:
    try:
        r = bbo.propose_ei(fid, up_to=PRIOR, seed=SEED)
        rows.append(dict(func=f"F{fid}", submitted=bbo.submission(proposals[fid]),
                         ei_would_have=r["submission"]))
    except Exception as e:
        rows.append(dict(func=f"F{fid}", submitted=bbo.submission(proposals[fid]),
                         ei_would_have=f"unavailable ({type(e).__name__})"))
pd.DataFrame(rows)


### Surrogate trust check

Run before reading any acquisition value, not after.


In [ ]:
# Is each surrogate worth listening to? LOO R2 < 0 means it is worse than
# predicting the mean, and any acquisition value built on it is arbitrary.
rows = []
for fid in bbo.FUNC_IDS:
    X, y, _ = bbo.load(fid, up_to=PRIOR)
    r2 = bbo.fit(fid, up_to=PRIOR).loo_r2() if len(y) >= 4 else float("nan")
    rows.append(dict(func=f"F{fid}", n_data=len(y), loo_r2=round(r2, 3),
                     verdict="broken" if r2 < 0 else "usable" if r2 == r2 else "too few points"))
pd.DataFrame(rows)


### Anchor audit


In [ ]:
ANCHOR = {
    1: [0.018957, 0.259878],
    2: [0.098559, 0.954719],
    3: [0.159998, 0.011915, 0.958587],
    4: [0.611147, 0.607958, 0.671974, 0.601141],
    5: [0.014852, 0.297741, 0.718557, 0.219953],
    6: [0.398747, 0.385554, 0.570014, 0.711777, 0.389141],
    7: [0.151858, 0.148558, 0.071547, 0.258484, 0.285157, 0.741141],
    8: [0.159174, 0.118198, 0.137956, 0.716535, 0.781515, 0.543548, 0.279585, 0.258543],
}
# Anchor audit: is each proposal being generated from the best point on record?
# This is the check whose absence cost the campaign most of its final score.
for fid in bbo.FUNC_IDS:
    w = bbo.anchor_check(fid, np.array(ANCHOR[fid], float), up_to=PRIOR)
    print(f"F{fid}: {w if w else 'anchored on best-known point'}")


## 3. Visualise

Best-so-far trajectory per function, truncated to the data available this round.


In [ ]:
import matplotlib
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(15, 6))
for ax, fid in zip(axes.ravel(), bbo.FUNC_IDS):
    try:
        _, y, rounds = bbo.load(fid, up_to=PRIOR)
    except ValueError:
        ax.set_title(f"F{fid}: no data"); continue
    ax.plot(rounds, y, "o", ms=4, alpha=.55)
    ax.plot(rounds, np.maximum.accumulate(y), "-", lw=2)
    ax.set_title(f"F{fid} (d={bbo.DIMS[fid]})", fontsize=9)
    ax.tick_params(labelsize=7); ax.set_xlabel("round", fontsize=8)
fig.suptitle(f"Best so far through round {PRIOR}", fontsize=11)
fig.tight_layout(); fig.savefig(f"{OUTDIR}/trajectories.png", dpi=140)
plt.show()


## 4. Submission strings


In [ ]:
# Portal format: six decimals, dash-separated, one line per function, no labels.
for fid in bbo.FUNC_IDS:
    print(bbo.submission(proposals[fid]))


## 5. After the portal returns each y

Returns recorded below and folded into `bbo.HISTORY` so the next round sees them.


In [ ]:
# Week 5 portal returns - already folded into bbo.HISTORY.
# returned_y = {
#     1: -3.8357016898973296e-154,
#     2: -0.10664658336026589,
#     3: -0.08523330692617447,
#     4: -4.082994951761037,
#     5: 2.379876233247608,
#     6: -2.000030696307407,
#     7: 0.16205712966379487,
#     8: 8.6790906222159,
# }
#
# F4 -4.083, its best for the next six rounds. F5 dropped to 2.38 - two orders below
# W1's 566.34, which at this point is still sitting unremarked in the record. The
# incumbent that later rounds anchor on is already the wrong one.
#
# for fid, y in returned_y.items():
#     bbo.append_result(fid, proposals[fid], y, rnd=WEEK)
